# 4. Batch prediction
Load the registered model, score the data, and write predictions back to UC. This notebook is what the scheduled job runs (see `resources/batch_inference.job.yml`).

**Concepts:** jobs, scheduling, loading a UC model, writing a Delta table.

In [ ]:
dbutils.widgets.text("catalog", "main")
dbutils.widgets.text("schema", "ml_workshop")
catalog = dbutils.widgets.get("catalog")
schema = dbutils.widgets.get("schema")
model_uri = f"models:/{catalog}.{schema}.taxi_fare/1"  # pin v1; use @champion alias in prod
output_table = f"{catalog}.{schema}.fare_predictions"

In [ ]:
import mlflow
mlflow.set_registry_uri("databricks-uc")

# Load the model and score in pandas — simple and clear for the workshop.
model = mlflow.pyfunc.load_model(model_uri)
pdf = spark.table("samples.nyctaxi.trips").toPandas()
pdf["predicted_fare"] = model.predict(pdf[["trip_distance"]])

In [ ]:
# Write back to UC as a Delta table. Overwrite keeps the demo simple.
# Enrichment: for large data, score distributed with `mlflow.pyfunc.spark_udf`
# instead of pandas; MERGE for incremental updates; or filter to new dates only.
spark.createDataFrame(pdf).write.mode("overwrite").saveAsTable(output_table)
print(f"Wrote {len(pdf)} predictions to {output_table}")

> **Expand here:** add job email/Slack notifications, retries, or a data-quality gate before writing.